# Redrob candidate-ranking sandbox
Run all cells to reproduce the CPU-only ranking pipeline. By default the notebook uses a bundled synthetic sample; set `USE_BUNDLED_SAMPLE = False` to upload a JSON, JSONL, or gzip candidate sample of your own.

The ranker gives the most weight to production career evidence in retrieval, ranking, evaluation, deployment, and ownership. It then combines role fit, credible skill depth, experience, product-company context, behavioral availability, and logistics. It uses no GPU, model download, external API, or network call during ranking.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path('/content/RedrobAI_hackathon')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run([
        'git', 'clone',
        'https://github.com/Man1ac-1773/RedrobAI_hackathon.git',
        str(repo),
    ], check=True)
os.chdir(repo)
print('Ranker ready:', repo)

In [ ]:
USE_BUNDLED_SAMPLE = True

if USE_BUNDLED_SAMPLE:
    candidate_path = str(repo / 'demo_candidates.jsonl')
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one candidate file')
    candidate_path = str(repo / next(iter(uploaded)))
print('Candidate input:', candidate_path)

In [ ]:
output_path = str(repo / 'sandbox_ranking.csv')
subprocess.run([
    sys.executable, 'preflight.py',
    '--candidates', candidate_path,
], check=True)
subprocess.run([
    sys.executable, 'rank.py',
    '--candidates', candidate_path,
    '--out', output_path,
], check=True)

In [ ]:
import csv

with open(output_path, encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
print(f'Generated {len(rows)} ranked rows')
for row in rows[:10]:
    print(f"#{row['rank']} {row['candidate_id']} score={row['score']}")
    print(row['reasoning'])

In [ ]:
DOWNLOAD_CSV = True
if DOWNLOAD_CSV:
    from google.colab import files
    files.download(output_path)